# 02b Narrative Sentiment

Precompute VADER sentiment on `cleaned_complaints.parquet` so the supervised notebooks can reuse a saved feature instead of recalculating it every run.

AI Assistance:
OpenAI ChatGPT was used for code debugging, code generation, code organization,
and code methodological brainstorming. All final modeling, implementation,
validation, commentary, and interpretation were performed and verified by the authors.


## Overview

This notebook reads the cleaned complaints dataset, computes a package-based sentiment score using VADER, and writes parquet outputs that can be reused by the final supervised notebooks.

Outputs:
- `data/processed/cleaned_complaints_vader.parquet`
- `data/processed/consumer_banking_relief_vader.parquet`


In [1]:
from pathlib import Path

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SOURCE_PATH = DATA_DIR / "final_unsupervised_features.parquet"
FULL_OUTPUT_PATH = DATA_DIR / "final_unsupervised_features_vader.parquet"
RELIEF_OUTPUT_PATH = DATA_DIR / "final_unsupervised_features_relief_vader.parquet"

assert SOURCE_PATH.exists(), f"Expected cleaned parquet at {SOURCE_PATH}"


In [2]:
df = pd.read_parquet(SOURCE_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {SOURCE_PATH.name}")
df.head()


Loaded 398,004 rows and 43 columns from final_unsupervised_features.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob
0,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,...,0.712191,0.000602,0.061553,0.000602,0.000602,0.000602,0.000602,0.000602,0.061924,0.000602
1,2020-04-10,Credit card or prepaid card,General-purpose prepaid card,Trouble using the card,Trouble getting information about the card,I got a Brinks Money pre-paid card in the mail...,Company has responded to the consumer and the ...,Netspend Corporation,IL,60657,...,0.002174,0.002174,0.002174,0.182673,0.002174,0.002174,0.002174,0.002174,0.002174,0.002174
2,2019-07-09,Credit card or prepaid card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,On XX/XX/XXXX I was called by a creditor Nelso...,NaN,Nelson Cruz & Associates LLC,TN,37043,...,0.000926,0.077052,0.000926,0.000926,0.000926,0.000926,0.037359,0.000926,0.000926,0.000926
3,2020-07-10,Credit card or prepaid card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,Around XX/XX/2020 i XXXX XXXX XXXX opened a cr...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,937XX,...,0.018011,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.043609
4,2019-06-24,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,Equifax sent me credit card suggestions to hel...,NaN,"EQUIFAX, INC.",OR,971XX,...,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083


## Sentiment Source

We use `cleaned_consumer_narrative` when it exists because it is the cleaned text field. If it is missing, we fall back to `Consumer complaint narrative`. Narratives represented as `[No Narrative]` receive a neutral score of `0.0`.


In [3]:
analyzer = SentimentIntensityAnalyzer()

sentiment_text_col = "cleaned_consumer_narrative" if "cleaned_consumer_narrative" in df.columns else "Consumer complaint narrative"
text_series = (
    df[sentiment_text_col]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

no_narrative_mask = text_series.eq("") | text_series.eq("[No Narrative]")
df["narrative_sentiment_score"] = text_series.map(
    lambda value: analyzer.polarity_scores(value)["compound"] if value and value != "[No Narrative]" else 0.0
).astype(float)

display(
    pd.DataFrame(
        {
            "sentiment_source_column": [sentiment_text_col],
            "rows": [len(df)],
            "no_narrative_rows": [int(no_narrative_mask.sum())],
            "sentiment_min": [df["narrative_sentiment_score"].min()],
            "sentiment_mean": [df["narrative_sentiment_score"].mean()],
            "sentiment_max": [df["narrative_sentiment_score"].max()],
        }
    )
)


,sentiment_source_column,rows,no_narrative_rows,sentiment_min,sentiment_mean,sentiment_max
0,cleaned_consumer_narrative,398004,0,-1.0,-0.113414,1.0


## Write Outputs

The full enriched dataset is saved first. Then we also write the relief-focused subset used by the final supervised notebooks so those notebooks can load a precomputed sentiment feature directly.


In [4]:
relief_values = {
    "Closed with monetary relief",
    "Closed with non-monetary relief",
    "Closed with explanation",
}

relief_df = df.loc[df["Company response to consumer"].isin(relief_values)].copy()

df.to_parquet(FULL_OUTPUT_PATH, index=False)
relief_df.to_parquet(RELIEF_OUTPUT_PATH, index=False)

print(f"Saved full sentiment-enriched data to {FULL_OUTPUT_PATH}")
print(f"Saved relief subset with sentiment to {RELIEF_OUTPUT_PATH}")
print(f"Relief subset rows: {len(relief_df):,}")


Saved full sentiment-enriched data to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\final_unsupervised_features_vader.parquet
Saved relief subset with sentiment to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\final_unsupervised_features_relief_vader.parquet
Relief subset rows: 397,575
